# Data Wrangling Pipeline

This notebook builds a complete data wrangling pipeline using Python and pandas. The goal is to take a messy dataset, inspect it, clean it, transform it, and produce a final analysis-ready dataset.

The workflow includes:

1. Creating or loading raw data
2. Inspecting the structure of the dataset
3. Identifying missing values and duplicates
4. Cleaning text fields
5. Converting data types
6. Handling missing values
7. Creating new useful features
8. Removing or flagging unusual values
9. Saving the cleaned dataset

This can be used as a portfolio-ready data science project because it shows a clear, organized, reproducible pipeline.

## 1. Import Libraries

We begin by importing the libraries used in the pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
np.random.seed(42)

## 2. Create a Sample Messy Dataset

In a real project, this section would usually load a CSV, Excel file, JSON file, API result, or database query. For this notebook, we create a small messy customer/order dataset so the full cleaning pipeline can run without needing an external file.

In [ ]:
raw_data = {
    "customer_id": [101, 102, 103, 104, 105, 105, 106, 107, 108, 109, 110, 111],
    "customer_name": [" Alice Smith ", "BOB JOHNSON", "carol white", "David Brown", "Eva Green", "Eva Green", None, "Frank Black", "Grace Hall", "Henry King", "Ivy Stone", "Jack Moore"],
    "email": ["alice@email.com", "bob@email", "carol@email.com", "david@email.com", "eva@email.com", "eva@email.com", "unknown", "frank@email.com", None, "henry@email.com", "ivy@email.com", "jack@email.com"],
    "order_date": ["2025-01-05", "2025/01/08", "Jan 10 2025", "2025-01-15", "2025-01-20", "2025-01-20", "2025-01-25", "bad_date", "2025-02-01", "2025-02-05", "2025-02-12", "2025-02-20"],
    "product_category": [" Electronics", "electronics", "Home", "home ", "Books", "Books", "Clothing", "clothing", "Electronics", "Home", "books", "Clothing"],
    "quantity": [1, 2, 1, 3, 2, 2, None, 1, 1000, 2, 1, 4],
    "unit_price": [799.99, 499.50, 150.00, None, 25.99, 25.99, 60.00, 80.00, 999.99, 175.00, -10.00, 45.00],
    "country": ["US", "usa", "United States", "US ", "Canada", "Canada", "USA", None, "US", "United States", "Canada", "US"]
}

raw_df = pd.DataFrame(raw_data)
raw_df

## 3. Initial Inspection

Before cleaning, we inspect the dataset. This step helps us understand column names, data types, missing values, duplicates, and unusual values.

In [ ]:
print("Shape:", raw_df.shape)
print("\nData types:")
print(raw_df.dtypes)
print("\nMissing values:")
print(raw_df.isna().sum())
print("\nDuplicate rows:", raw_df.duplicated().sum())

## 4. Standardize Column Names

Clean column names make the dataset easier to work with. The columns are already mostly clean, but this step is useful in real projects where columns may contain spaces, capitals, or symbols.

In [ ]:
df = raw_df.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df.head()

## 5. Clean Text Fields

Text fields often contain inconsistent capitalization, extra spaces, and different spellings. We standardize names, categories, and countries.

In [ ]:
df["customer_name"] = df["customer_name"].str.strip().str.title()
df["product_category"] = df["product_category"].str.strip().str.title()

country_map = {
    "usa": "United States",
    "us": "United States",
    "united states": "United States",
    "canada": "Canada"
}

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(country_map)
)

df[["customer_name", "product_category", "country"]].head(10)

## 6. Convert Data Types

The order date should be a date column. Invalid dates are converted to missing values so they can be handled explicitly.

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

df.dtypes

## 7. Create Data Quality Flags

Instead of immediately deleting unusual records, it is often better to flag them first. This preserves transparency and makes the cleaning decisions easier to explain.

In [ ]:
df["invalid_email_flag"] = ~df["email"].astype("string").str.contains(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", regex=True, na=False)
df["missing_date_flag"] = df["order_date"].isna()
df["negative_price_flag"] = df["unit_price"] < 0
df["unusual_quantity_flag"] = df["quantity"] > 100

df[["email", "order_date", "quantity", "unit_price", "invalid_email_flag", "missing_date_flag", "negative_price_flag", "unusual_quantity_flag"]]

## 8. Handle Missing and Invalid Values

Cleaning choices should depend on the project. Here we use simple, explainable rules:

- Missing customer names become `Unknown Customer`
- Missing countries become `Unknown`
- Missing quantities become the median quantity
- Missing unit prices become the median unit price within the product category
- Negative prices are treated as invalid and replaced
- Extremely unusual quantities are capped

In [ ]:
df["customer_name"] = df["customer_name"].fillna("Unknown Customer")
df["country"] = df["country"].fillna("Unknown")

median_quantity = df.loc[df["quantity"] <= 100, "quantity"].median()
df["quantity"] = df["quantity"].fillna(median_quantity)

# Replace negative prices with missing first
df.loc[df["unit_price"] < 0, "unit_price"] = np.nan

# Fill unit price using category median, then overall median if needed
df["unit_price"] = df.groupby("product_category")["unit_price"].transform(lambda x: x.fillna(x.median()))
df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())

# Cap extreme quantity values at the 95th percentile of reasonable values
cap_value = df.loc[df["quantity"] <= 100, "quantity"].quantile(0.95)
df.loc[df["quantity"] > cap_value, "quantity"] = cap_value

df[["customer_name", "country", "quantity", "unit_price"]]

## 9. Remove Duplicate Records

Duplicate rows can distort totals and model results. Here we remove exact duplicate records.

In [ ]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)

## 10. Feature Engineering

After cleaning, we create new useful variables that can support analysis or modeling.

In [ ]:
df["total_sales"] = df["quantity"] * df["unit_price"]
df["order_month"] = df["order_date"].dt.to_period("M").astype("string")
df["order_day_of_week"] = df["order_date"].dt.day_name()

df[["customer_id", "product_category", "quantity", "unit_price", "total_sales", "order_month", "order_day_of_week"]].head()

## 11. Final Data Quality Check

Now we check whether the cleaned dataset is ready for analysis.

In [ ]:
quality_summary = pd.DataFrame({
    "column": df.columns,
    "missing_values": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
    "data_type": df.dtypes.astype(str).values
})

quality_summary

## 12. Simple Analysis After Cleaning

A good wrangling project should show that the cleaned data can now be used for meaningful analysis.

In [ ]:
category_sales = (
    df.groupby("product_category", as_index=False)["total_sales"]
    .sum()
    .sort_values("total_sales", ascending=False)
)

category_sales

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(category_sales["product_category"], category_sales["total_sales"])
plt.title("Total Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 13. Save the Cleaned Dataset

The final step is to save the cleaned dataset so it can be used in later analysis, dashboards, or machine learning models.

In [ ]:
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

clean_file = output_folder / "cleaned_customer_orders.csv"
df.to_csv(clean_file, index=False)

print("Cleaned dataset saved to:", clean_file)

## 14. Conclusion

This notebook demonstrated a complete data wrangling pipeline. The raw dataset contained common real-world problems, including inconsistent text formatting, missing values, duplicate records, invalid dates, invalid emails, negative prices, and unusual quantities.

The final cleaned dataset is more consistent, more reliable, and ready for analysis. The project shows that data wrangling is not just about fixing errors. It is the foundation that makes meaningful analysis and modeling possible.